# FEC Donations → Legislator Matching Pipeline

Matches FEC small-dollar donation data to the `unitedstates/congress` legislators JSON,
producing a merged CSV with bioguide IDs, Twitter handles, chamber, state, and party.

**Steps**
1. Load data
2. Clean FEC names
3. Build legislator lookup indexes
4. Auto-match (fuzzy)
5. Review & apply manual overrides
6. Export final CSV

**Data sources**
- FEC: https://www.fec.gov/data/
- Legislators JSON: https://github.com/unitedstates/congress-legislators

---
## 0 · Imports & config

In [1]:
import json
import re
from collections import defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

# Optional: rapidfuzz is much faster than difflib for large lists
# Install with:  pip install rapidfuzz
try:
    from rapidfuzz import fuzz as _rf
    def token_sort_ratio(a, b):
        return _rf.token_sort_ratio(a, b)
    print("Fuzzy backend: rapidfuzz")
except ImportError:
    def token_sort_ratio(a, b):
        a_s = " ".join(sorted(a.lower().split()))
        b_s = " ".join(sorted(b.lower().split()))
        return SequenceMatcher(None, a_s, b_s).ratio() * 100
    print("Fuzzy backend: difflib (pip install rapidfuzz for speed)")

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 100)

Fuzzy backend: rapidfuzz


In [2]:
# ── File paths — edit these to match your local setup ──────────────────────
FEC_CSV         = Path("donation_proportions.csv")
CURRENT_JSON    = Path("legislators-current.json")
HISTORICAL_JSON = Path("legislators-historical.json")
OUTPUT_CSV      = Path("fec_legislators_matched.csv")

# Minimum fuzzy score (0–100) to accept an automatic match.
# Lower = more matches but more false positives. 70 is a reasonable start.
AUTO_MATCH_THRESHOLD = 70

---
## 1 · Load data

In [3]:
fec_raw = pd.read_csv(FEC_CSV)
print(f"FEC rows: {len(fec_raw)}")
fec_raw.head()

FEC rows: 538


,Name,Total Raised,Total from Small Donors,Percent from Small Donors*,total money raised
0,TaylorMarjorie Taylor Greene (R),$7932166,$5763400,72.66%,Raised over $100k
1,Ocasio-CortezAlexandria Ocasio-Cortez (D),$15160101,$10602414,69.94%,Raised over $100k
2,SandersBernie Sanders (I),$35728405,$22732498,63.63%,Raised over $100k
3,JordanJim Jordan (R),$13313614,$7423207,55.76%,Raised over $100k
4,CraneEli Crane (R),$8437397,$4638205,54.97%,Raised over $100k


In [4]:
with open(CURRENT_JSON) as f:
    current_members = json.load(f)

with open(HISTORICAL_JSON) as f:
    historical_members = json.load(f)

print(f"Current members  : {len(current_members)}")
print(f"Historical members: {len(historical_members)}")

Current members  : 538
Historical members: 12225


---
## 2 · Clean FEC names

The raw FEC `Name` field looks like `"TaylorMarjorie Taylor Greene (R)"` —
the last name is concatenated onto the front, then the full name follows, then the party in parentheses.

We need to:
1. Extract the **party code** from the trailing `(R)` / `(D)` / `(I)`
2. Strip the run-together last-name prefix to recover a **clean full name**

The prefix strip happens later during matching (we use each legislator's last name
as the candidate prefix), so here we just do light cleanup.

In [5]:
PARTY_MAP = {"R": "Republican", "D": "Democrat", "I": "Independent"}

fec = fec_raw.copy()

# Extract party code and full party name
fec["party_code"] = fec["Name"].str.extract(r"\(([RDI])\)")[0]
fec["party_full"] = fec["party_code"].map(PARTY_MAP)

# Strip trailing whitespace from all string columns
fec["Name"] = fec["Name"].str.strip()

# Convert donation columns to numeric (remove $ and commas)
for col in ["Total Raised", "Total from Small Donors"]:
    if col in fec.columns:
        fec[col + "_num"] = (
            fec[col].astype(str)
            .str.replace(r"[$,]", "", regex=True)
            .pipe(pd.to_numeric, errors="coerce")
        )

# Convert percent to float
pct_col = "Percent from Small Donors*"
if pct_col in fec.columns:
    fec["pct_small_donors"] = (
        fec[pct_col].astype(str)
        .str.replace("%", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

print(f"Rows: {len(fec)}")
print(f"Party breakdown:")
print(fec["party_full"].value_counts())
print()
fec[["Name", "party_code", "party_full", "pct_small_donors"]].head(10)

Rows: 538
Party breakdown:
party_full
Republican     271
Democrat       262
Independent      4
Name: count, dtype: int64



,Name,party_code,party_full,pct_small_donors
0,TaylorMarjorie Taylor Greene (R),R,Republican,72.66
1,Ocasio-CortezAlexandria Ocasio-Cortez (D),D,Democrat,69.94
2,SandersBernie Sanders (I),I,Independent,63.63
3,JordanJim Jordan (R),R,Republican,55.76
4,CraneEli Crane (R),R,Republican,54.97
5,WarrenElizabeth Warren (D),D,Democrat,54.62
6,PelosiNancy Pelosi (D),D,Democrat,53.39
7,PaulRand Paul (R),R,Republican,51.98
8,SpartzVictoria Spartz (R),R,Republican,50.38
9,BurgessMichael Burgess (R),R,Republican,50.35


In [6]:
# Sanity check: any rows with no party code?
no_party = fec[fec["party_code"].isna()]
print(f"Rows with no party code: {len(no_party)}")
if len(no_party):
    display(no_party[["Name", "party_code"]])

Rows with no party code: 1


,Name,party_code
528,GonzalezJenniffer Gonzalez (3),NaN


---
## 3 · Build legislator lookup indexes

We build two indexes:

- **`last_name_index`** — keyed on lowercase last name → list of current member dicts.
  Used for auto-matching (current congress only).
- **`bioguide_index`** — keyed on bioguide ID → member dict.
  Spans current + historical, used for manual override lookups
  (so members who left congress mid-cycle, like Santos, are still found).

In [7]:
def get_official_name(member):
    n = member["name"]
    return n.get("official_full") or f"{n.get('first', '')} {n.get('last', '')}".strip()


def member_to_row(member):
    """Flatten a legislator JSON entry to a plain dict."""
    if not member.get("terms"):
        return None
    last_term = member["terms"][-1]
    return {
        "first":        member["name"].get("first", ""),
        "last":         member["name"].get("last", ""),
        "official_full": get_official_name(member),
        "party":        last_term.get("party", ""),
        "chamber":      last_term.get("type", ""),
        "state":        last_term.get("state", ""),
        "bioguide":     member["id"].get("bioguide", ""),
        "twitter":      member["id"].get("twitter", ""),
    }


# Auto-matcher index — current congress only
current_rows = [r for m in current_members if (r := member_to_row(m))]

last_name_index = defaultdict(list)
for row in current_rows:
    last_name_index[row["last"].lower()].append(row)

# Override lookup index — current + historical
bioguide_index = {}
for m in current_members + historical_members:
    bid = m["id"].get("bioguide")
    if not bid or not m.get("terms"):
        continue
    last_term = m["terms"][-1]
    bioguide_index[bid] = {
        "official_full": get_official_name(m),
        "party":   last_term.get("party", ""),
        "chamber": last_term.get("type", ""),
        "state":   last_term.get("state", ""),
        "bioguide": bid,
        "twitter": m["id"].get("twitter", ""),
    }

print(f"Current member rows : {len(current_rows)}")
print(f"Last-name index keys: {len(last_name_index)}")
print(f"Bioguide index size : {len(bioguide_index)}")

Current member rows : 538
Last-name index keys: 489
Bioguide index size : 12763


---
## 4 · Auto-match (fuzzy)

For each FEC row we:
1. Try every legislator last name as a candidate prefix for the run-together FEC name.
2. Strip it to recover a clean name, then score against `official_full` with token-sort fuzzy ratio.
3. Accept the best score above `AUTO_MATCH_THRESHOLD`.

In [8]:
def auto_match_one(raw, party, last_name_index, threshold):
    """
    Returns (best_candidate_dict, score, clean_name) or (None, 0, None).
    """
    best_match = None
    best_score = 0.0
    best_clean = None

    for last, candidates in last_name_index.items():
        if not raw.lower().startswith(last.lower()):
            continue

        # Strip the prefix and the trailing party code
        pat = re.escape(last)
        m = re.match(rf"^{pat}(.+?)\s*\([RDI3]\)\s*$", raw, re.IGNORECASE)
        if not m:
            continue
        clean = m.group(1).strip()

        for cand in candidates:
            if isinstance(party, str) and cand["party"] != party:
                continue
            score = token_sort_ratio(clean, cand["official_full"])
            if score > best_score:
                best_score = score
                best_match = cand
                best_clean = clean

    if best_match and best_score >= threshold:
        return best_match, round(best_score, 1), best_clean
    return None, round(best_score, 1), best_clean


# Run auto-matching on all FEC rows
auto_results = []
for row in fec.to_dict("records"):
    cand, score, clean = auto_match_one(
        row["Name"], row["party_full"], last_name_index, AUTO_MATCH_THRESHOLD
    )
    auto_results.append({
        **row,
        **(cand or {}),
        "clean_name":   clean,
        "match_score":  score,
        "match_method": "auto" if cand else "no_match",
    })

auto_df = pd.DataFrame(auto_results)
auto_matched   = auto_df[auto_df["match_method"] == "auto"]
auto_unmatched = auto_df[auto_df["match_method"] == "no_match"]

print(f"Auto-matched : {len(auto_matched)}")
print(f"Unmatched    : {len(auto_unmatched)}")

Auto-matched : 429
Unmatched    : 109


In [9]:
# ── Inspect low-confidence auto-matches — decide whether to keep or override
LOW_CONFIDENCE_THRESHOLD = 85

low_conf = auto_matched[auto_matched["match_score"] < LOW_CONFIDENCE_THRESHOLD].copy()
low_conf = low_conf.sort_values("match_score")

print(f"Low-confidence matches (score < {LOW_CONFIDENCE_THRESHOLD}): {len(low_conf)}")
low_conf[["Name", "clean_name", "official_full", "match_score"]]

Low-confidence matches (score < 85): 33


,Name,clean_name,official_full,match_score
528,GonzalezJenniffer Gonzalez (3),Jenniffer Gonzalez,Vicente Gonzalez,70.6
350,CarterBuddy Carter (R),Buddy Carter,"Earl L. ""Buddy"" Carter",70.6
319,CoonsChris Coons (D),Chris Coons,Christopher A. Coons,71.0
142,SmithChris Smith (R),Chris Smith,Christopher H. Smith,71.0
232,KingAngus King (I),Angus King,"Angus S. King, Jr.",71.4
92,MarkeyEd Markey (D),Ed Markey,Edward J. Markey,72.0
518,FleischmannChuck Fleischmann (R),Chuck Fleischmann,"Charles J. ""Chuck"" Fleischmann",72.3
436,WeberRandy Weber (R),Randy Weber,"Randy K. Weber, Sr.",73.3
123,KiggansJen Kiggans (R),Jen Kiggans,Jennifer A. Kiggans,73.3
488,SimpsonMike Simpson (R),Mike Simpson,Michael K. Simpson,73.3


In [10]:
# ── Inspect unmatched rows — these need manual overrides or are true non-members
print(f"Unmatched rows: {len(auto_unmatched)}")
auto_unmatched[["Name", "party_full", "match_score"]]

Unmatched rows: 109


,Name,party_full,match_score
0,TaylorMarjorie Taylor Greene (R),Republican,48.6
9,BurgessMichael Burgess (R),Republican,0.0
24,AllredColin Allred (D),Democrat,0.0
29,PaulinaAnna Paulina Luna (R),Republican,48.3
30,CortezCatherine Cortez Masto (D),Democrat,0.0
...,...,...,...
531,BucshonLarry Bucshon (R),Republican,0.0
532,McHenryPatrick McHenry (R),Republican,0.0
533,PenceGreg Pence (R),Republican,0.0
535,SablanGregorio Sablan (I),Independent,0.0


---
## 5 · Manual overrides

Add bioguide IDs here for any row that the auto-matcher missed or got wrong.
Copy the exact `Name` string from the unmatched table above as the key.

Setting a value to `None` marks a row as a **known non-member** (e.g. a candidate
who never served) — it will still appear in the output with `match_method = "known_non_member"`
so you can filter it later if you want.

Bioguide IDs come from https://github.com/unitedstates/congress-legislators

In [11]:
MANUAL_OVERRIDES = {
    # ── Unmatched because of compound / hyphenated last names ────────────
    "TaylorMarjorie Taylor Greene (R)":       "G000596",
    "BurgessMichael Burgess (R)":             "B001248",
    "AllredColin Allred (D)":                 "A000378",
    "CortezCatherine Cortez Masto (D)":       "C001113",
    "RubioMarco Rubio (R)":                   "R000595",
    "DeMonica De La Cruz (R)":                "D000605",
    "StabenowDebbie Stabenow (D)":            "S000770",
    "PorterKatie Porter (D)":                 "P000618",
    "PeltolaMary Peltola (D)":                "P000619",
    "BushCori Bush (D)":                      "B001324",
    "JacksonJeff Jackson (D)":                "J000308",
    "TesterJon Tester (D)":                   "T000464",
    "BrownSherrod Brown (D)":                 "B000944",
    "GrijalvaRaul M Grijalva (D)":            "G000551",
    "HassanMaggie Hassan (D)":                "H001076",
    "VanDerrick Van Orden (R)":               "V000135",
    "VanJeff Van Drew (R)":                   "V000133",
    "BowmanJamaal Bowman (D)":                "B001223",
    "CaseyBob Casey (D)":                     "C001070",
    "VanceJ D Vance (R)":                     "V000137",
    "RayBen Ray Lujan (D)":                   "L000570",
    "WaltzMichael Waltz (R)":                 "W000823",
    "LeeBarbara Lee (D)":                     "L000551",
    "SpanbergerAbigail Spanberger (D)":       "S001209",
    "GoodBob Good (R)":                       "G000595",
    "WextonJennifer Wexton (D)":              "W000825",
    "DuncanJeff Duncan (R)":                  "D000615",
    "MooneyAlex Mooney (R)":                  "M001195",
    "SteelMichelle Steel (R)":                "S001135",
    "CarlJerry Carl (R)":                     "C001114",
    "CaraveoYadira Caraveo (D)":              "C001134",
    "SarbanesJohn Sarbanes (D)":              "S001168",
    "GarciaMike Garcia (R)":                  "G000598",
    "WildSusan Wild (D)":                     "W000826",
    "ConnollyGerry Connolly (D)":             "C001078",
    "DurbinDick Durbin (D)":                  "D000563",
    "SherrillMikie Sherrill (D)":             "S001207",
    "MolinaroMarc Molinaro (R)":              "M001226",
    "PoseyBill Posey (R)":                    "P000599",
    "LeskoDebbie Lesko (R)":                  "L000589",
    "BraunMike Braun (R)":                    "B001310",
    "RosendaleMatt Rosendale (R)":            "R000103",
    "BluntLisa Blunt Rochester (D)":          "B001303",
    "GluesenkampMarie Gluesenkamp Perez (D)": "G000601",
    "TiffanyTom Tiffany (R)":                 "T000165",
    "LegerTeresa Leger Fernandez (D)":        "L000579",
    "Chavez-DeremerLori Chavez-Deremer (R)":  "C001138",
    "McMorrisCathy McMorris Rodgers (R)":     "M001159",
    "GayMary Gay Scanlon (D)":                "S001205",
    "WassermanDebbie Wasserman Schultz (D)":  "W000797",
    "WatsonBonnie Watson Coleman (D)":        "W000822",
    "VanBeth Van Duyne (R)":                  "V000136",
    "VanChris Van Hollen (D)":                "V000128",
    "MooreShelley Moore Capito (R)":          "C001047",
    "LaMalfaDoug LaMalfa (R)":                "L000578",
    "LuetkemeyerBlaine Luetkemeyer (R)":      "L000569",
    "HolmesEleanor Holmes Norton (D)":        "N000147",
    "RuppersbergerDutch Ruppersberger (D)":   "R000576",
    "PaulinaAnna Paulina Luna (R)":           "L000603",
    "GreenMark Green (R)":                    "G000590",
    "KilmerDerek Kilmer (D)":                 "K000381",
    "LaturnerJake Laturner (R)":              "L000600",
    "WenstrupBrad Wenstrup (R)":              "W000815",
    "KeatingBill Keating (D)":                "K000375",
    "SinemaKyrsten Sinema (I)":               "S001191",
    "RomneyMitt Romney (R)":                  "R000615",
    "CartwrightMatt Cartwright (D)":          "C001090",
    "KusterAnn Kuster (D)":                   "K000382",
    "BeyerDon Beyer (D)":                     "B001292",
    "JohnsonHank Johnson (D)":                "J000288",
    "BishopDan Bishop (R)":                   "B001311",
    "NickelWiley Nickel (D)":                 "N000193",
    "PhillipsDean Phillips (D)":              "P000616",
    "KildeeDan Kildee (D)":                   "K000380",
    "CarperTom Carper (D)":                   "C000174",
    "HimesJim Himes (D)":                     "H001047",
    "GarciaJesus Garcia (D)":                 "G000586",
    "VelazquezNydia Velazquez (D)":           "V000081",
    "ScottBobby Scott (D)":                   "S000185",
    "LattaBob Latta (R)":                     "L000566",
    "ManchinJoe Manchin (D)":                 "M001183",
    "DuarteJohn Duarte (R)":                  "D000623",
    "EshooAnna Eshoo (D)":                    "E000215",
    "SanchezLinda Sanchez (D)":               "S001156",
    "LambornDoug Lamborn (R)":                "L000564",
    "KamlagerSydney Kamlager (D)":            "K000400",
    "CrawfordRick Crawford (R)":              "C001087",
    "BlumenauerEarl Blumenauer (D)":          "B000574",
    "ArmstrongKelly Armstrong (R)":           "A000377",
    "GravesGarret Graves (R)":                "G000577",
    "FergusonDrew Ferguson (R)":              "F000465",
    "CardenasTony Cardenas (D)":              "C001097",
    "TroneDavid Trone (D)":                   "T000468",
    "GrangerKay Granger (R)":                 "G000377",
    "BucshonLarry Bucshon (R)":               "B001275",
    "McHenryPatrick McHenry (R)":             "M001156",
    "PenceGreg Pence (R)":                    "P000615",
    "SablanGregorio Sablan (I)":              "S001177",
    "NapolitanoGrace Napolitano (D)":         "N000179",
    "CardinBen Cardin (D)":                   "C000141",
    "ManningKathy Manning (D)":               "M001210",
    "LeeErica Lee Carter (D)":                "C001049",
    "BairdJim Baird (R)":                     "B001307",
    "BarraganNanette Barragan (D)":           "B001300",
    # ── Served in congress but left before end of term ───────────────────
    "SantosGeorge Santos (R)":                "S001211",  # expelled Nov 2023
    # ── Candidates who never served — kept in output for completeness ─────
    "LopezGreg Lopez (R)":                    None,       # candidate only
}

In [12]:
# Apply overrides on top of the auto-match results
final_rows = []

for row in auto_df.to_dict("records"):
    raw = row["Name"]

    if raw not in MANUAL_OVERRIDES:
        # Keep auto result as-is
        final_rows.append(row)
        continue

    bid = MANUAL_OVERRIDES[raw]

    if bid is None:
        # Known non-member candidate — keep row but flag it
        final_rows.append({
            **row,
            "bioguide": None, "twitter": None,
            "chamber": None, "state": None,
            "official_full": None,
            "match_score": 0.0,
            "match_method": "known_non_member",
        })
        continue

    if bid not in bioguide_index:
        print(f"WARNING: bioguide {bid!r} not found for {raw!r} — check the ID")
        final_rows.append({
            **row,
            "match_method": "override_id_missing",
        })
        continue

    cand = bioguide_index[bid]
    final_rows.append({
        **row,
        **cand,
        "clean_name":   cand["official_full"],
        "match_score":  100.0,
        "match_method": "manual",
    })

final_df = pd.DataFrame(final_rows)

# Summary
print("Match method breakdown:")
print(final_df["match_method"].value_counts())
print()
matched_final   = final_df[final_df["bioguide"].notna()]
unmatched_final = final_df[final_df["bioguide"].isna()]
print(f"✅ Matched  : {len(matched_final)} / {len(final_df)} ({100*len(matched_final)/len(final_df):.1f}%)")
print(f"❌ Unmatched: {len(unmatched_final)}")

Match method breakdown:
match_method
auto                   429
manual                 104
no_match                 3
known_non_member         1
override_id_missing      1
Name: count, dtype: int64

✅ Matched  : 533 / 538 (99.1%)
❌ Unmatched: 5


In [13]:
# Show anything still unmatched — copy the Name strings into MANUAL_OVERRIDES above to fix
if len(unmatched_final):
    print("Still unmatched — add to MANUAL_OVERRIDES:")
    display(unmatched_final[["Name", "party_full", "match_score", "match_method"]])
else:
    print("All rows matched!")

Still unmatched — add to MANUAL_OVERRIDES:


,Name,party_full,match_score,match_method
172,WilliamsBrandon Williams (R),Republican,66.7,no_match
186,LopezGreg Lopez (R),Republican,0.0,known_non_member
194,Chavez-DeremerLori Chavez-Deremer (R),Republican,0.0,override_id_missing
296,D'EspositoAnthony D'Esposito (R),Republican,0.0,no_match
527,ColemanAmata Coleman Radewagen (R),Republican,35.3,no_match


---
## 6 · Export final CSV

In [14]:
# Select and rename output columns
out_cols = [
    "Name",
    "clean_name",
    "official_full",
    "bioguide",
    "twitter",
    "chamber",
    "state",
    "party",
    "party_code",
    "match_score",
    "match_method",
    "Total Raised",
    "Total Raised_num",
    "Total from Small Donors",
    "Total from Small Donors_num",
    "Percent from Small Donors*",
    "pct_small_donors",
    "total money raised",
]

out = final_df[[c for c in out_cols if c in final_df.columns]].copy()
out.to_csv(OUTPUT_CSV, index=False)

print(f"Saved → {OUTPUT_CSV}  ({len(out)} rows, {len(out.columns)} columns)")
out.head()

Saved → fec_legislators_matched.csv  (538 rows, 18 columns)


,Name,clean_name,official_full,bioguide,twitter,chamber,state,party,party_code,match_score,match_method,Total Raised,Total Raised_num,Total from Small Donors,Total from Small Donors_num,Percent from Small Donors*,pct_small_donors,total money raised
0,TaylorMarjorie Taylor Greene (R),Marjorie Taylor Greene,Marjorie Taylor Greene,G000596,,rep,GA,Republican,R,100.0,manual,$7932166,7932166,$5763400,5763400,72.66%,72.66,Raised over $100k
1,Ocasio-CortezAlexandria Ocasio-Cortez (D),Alexandria Ocasio-Cortez,Alexandria Ocasio-Cortez,O000172,,rep,NY,Democrat,D,100.0,auto,$15160101,15160101,$10602414,10602414,69.94%,69.94,Raised over $100k
2,SandersBernie Sanders (I),Bernie Sanders,Bernard Sanders,S000033,,sen,VT,Independent,I,82.8,auto,$35728405,35728405,$22732498,22732498,63.63%,63.63,Raised over $100k
3,JordanJim Jordan (R),Jim Jordan,Jim Jordan,J000289,,rep,OH,Republican,R,100.0,auto,$13313614,13313614,$7423207,7423207,55.76%,55.76,Raised over $100k
4,CraneEli Crane (R),Eli Crane,Elijah Crane,C001132,,rep,AZ,Republican,R,85.7,auto,$8437397,8437397,$4638205,4638205,54.97%,54.97,Raised over $100k


In [15]:
# Quick summary stats on the matched data
matched_out = out[out["bioguide"].notna()]

print("=== Chamber breakdown ===")
print(matched_out["chamber"].value_counts())
print()
print("=== Party breakdown ===")
print(matched_out["party"].value_counts())
print()
print("=== Small donor % distribution ===")
print(matched_out["pct_small_donors"].describe().round(1))
print()
print("=== Twitter handle coverage ===")
has_twitter = matched_out["twitter"].notna() & (matched_out["twitter"] != "")
print(f"{has_twitter.sum()} / {len(matched_out)} members have a Twitter handle ({100*has_twitter.mean():.0f}%)")

=== Chamber breakdown ===
chamber
rep    427
sen    106
Name: count, dtype: int64

=== Party breakdown ===
party
Democrat       265
Republican     264
Independent      4
Name: count, dtype: int64

=== Small donor % distribution ===
count    533.0
mean      12.2
std       13.7
min        0.0
25%        2.3
50%        6.6
75%       16.2
max       72.7
Name: pct_small_donors, dtype: float64

=== Twitter handle coverage ===
0 / 533 members have a Twitter handle (0%)


In [16]:
# Load social media file and merge Twitter handles
with open("legislators-social-media.json") as f:
    social = json.load(f)

# Build bioguide -> twitter lookup
twitter_lookup = {}
for m in social:
    bid = m["id"].get("bioguide")
    handle = m.get("social", {}).get("twitter")
    if bid and handle:
        twitter_lookup[bid] = handle

# Apply to matched dataframe
final_df["twitter"] = final_df["bioguide"].map(twitter_lookup)

# Check coverage
has_twitter = final_df["twitter"].notna()
print(f"Twitter handles found: {has_twitter.sum()} / {len(final_df)}")

# Re-export
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"Updated → {OUTPUT_CSV}")

Twitter handles found: 461 / 538
Updated → fec_legislators_matched.csv
